<a href="https://colab.research.google.com/github/julschleinitz/ai4chemistry-bootcamp/blob/main/tutorials/molecular-representations.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Tutorial 2 — Molecular Property Prediction
## AI for Chemical Sciences Bootcamp · Caltech, August 2026

**Instructor:** Jules Schleinitz  
**Estimated time:** 90 min

---

### What you will learn

How you encode a molecule determines what a model can learn. In this tutorial we compare four representations — from handcrafted descriptors to learned graph embeddings — and build a property prediction pipeline on the **ESOL aqueous solubility** dataset.

| Step | Representation | Model |
|------|---------------|-------|
| 1 | RDKit physicochemical descriptors | Ridge regression |
| 2 | Morgan fingerprints (ECFP4) | Random Forest |
| 3 | Morgan fingerprints | Feed-forward neural network |
| 4 | Molecular graph (atoms + bonds) | Message-passing GNN |

By the end you will have trained all four models on the same task and understand *why* each representation encodes different information.

---
## 0. Setup

In [ ]:
# Install dependencies (Colab only — comment out if running locally)
!pip install rdkit-pypi torch torch-geometric deepchem pandas scikit-learn matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# RDKit
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D
from rdkit import DataStructs

# Scikit-learn
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# PyTorch Geometric
from torch_geometric.data import Data, DataLoader as GeoLoader
from torch_geometric.nn import GCNConv, global_mean_pool

print("All imports OK")

---
## 1. Dataset — ESOL aqueous solubility

The **ESOL** dataset (Delaney, 2004) contains measured aqueous solubility (log mol/L) for 1,128 small organic molecules. It is one of the most widely used benchmarks for molecular property prediction.

**Target:** `measured log(solubility:mol/L)` — a continuous value, so this is a regression task.

> **Why solubility?** It is directly relevant to drug formulation and agrochemistry, and it depends on a mix of electronic, steric, and hydrogen-bonding features — a good test for different representations.

In [ ]:
# Load ESOL directly from DeepChem's MoleculeNet
import deepchem as dc

tasks, datasets, transformers = dc.molnet.load_delaney(featurizer='Raw', splitter='scaffold')
train_ds, val_ds, test_ds = datasets

# Convert to a flat DataFrame for easier manipulation
def dataset_to_df(ds):
    smiles = [x.smiles if hasattr(x, 'smiles') else str(x) for x in ds.X]
    y = ds.y.flatten()
    return pd.DataFrame({'smiles': smiles, 'logS': y})

df_train = dataset_to_df(train_ds)
df_val   = dataset_to_df(val_ds)
df_test  = dataset_to_df(test_ds)
df_all   = pd.concat([df_train, df_val, df_test], ignore_index=True)

print(f"Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}")
df_train.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df_all['logS'], bins=40, color='#5DDAB4', edgecolor='white', linewidth=0.4)
axes[0].set_xlabel('log(S) [mol/L]')
axes[0].set_ylabel('Count')
axes[0].set_title('ESOL — solubility distribution')

# Molecular weight vs solubility
mws = []
for smi in df_all['smiles']:
    mol = Chem.MolFromSmiles(smi)
    mws.append(Descriptors.MolWt(mol) if mol else np.nan)
axes[1].scatter(mws, df_all['logS'], alpha=0.4, s=12, c='#5DDAB4')
axes[1].set_xlabel('Molecular weight (Da)')
axes[1].set_ylabel('log(S) [mol/L]')
axes[1].set_title('MW vs solubility')

plt.tight_layout()
plt.show()

In [ ]:
# Visualise a few molecules from each solubility range
df_sorted = df_all.sort_values('logS')
examples = pd.concat([
    df_sorted.head(3),            # poorly soluble
    df_sorted.iloc[len(df_sorted)//2 - 1 : len(df_sorted)//2 + 2],  # medium
    df_sorted.tail(3)             # highly soluble
])

mols  = [Chem.MolFromSmiles(s) for s in examples['smiles']]
labels = [f"logS = {v:.2f}" for v in examples['logS']]
img = Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(300, 200), legends=labels)
img

---
## 2. Representation 1 — Physicochemical descriptors

The simplest approach: compute a fixed set of expert-designed features from the 2D structure.

RDKit exposes ~200 descriptors. We will use a curated subset that directly captures solubility-relevant properties:

| Descriptor | Chemical meaning |
|-----------|------------------|
| `MolLogP` | Lipophilicity (Wildman-Crippen) |
| `MolWt` | Molecular weight |
| `NumHDonors` / `NumHAcceptors` | H-bond capacity |
| `TPSA` | Topological polar surface area |
| `NumRotatableBonds` | Flexibility |
| `RingCount` | Aromaticity proxy |

These correspond to the features used in Delaney's original rule-based model.

In [ ]:
DESCRIPTOR_FNS = {
    'MolLogP':           Descriptors.MolLogP,
    'MolWt':             Descriptors.MolWt,
    'NumHDonors':        rdMolDescriptors.CalcNumHBD,
    'NumHAcceptors':     rdMolDescriptors.CalcNumHBA,
    'TPSA':              Descriptors.TPSA,
    'NumRotatableBonds': rdMolDescriptors.CalcNumRotatableBonds,
    'RingCount':         Descriptors.RingCount,
    'FractionCSP3':      rdMolDescriptors.CalcFractionCSP3,
}

def compute_descriptors(smiles_list):
    rows = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            rows.append([np.nan] * len(DESCRIPTOR_FNS))
        else:
            rows.append([fn(mol) for fn in DESCRIPTOR_FNS.values()])
    return pd.DataFrame(rows, columns=list(DESCRIPTOR_FNS.keys()))

X_desc_train = compute_descriptors(df_train['smiles'])
X_desc_val   = compute_descriptors(df_val['smiles'])
X_desc_test  = compute_descriptors(df_test['smiles'])

print(X_desc_train.shape)
X_desc_train.head()

In [ ]:
# Correlation of each descriptor with logS
corr_df = X_desc_train.copy()
corr_df['logS'] = df_train['logS'].values
corr = corr_df.corr()['logS'].drop('logS').sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#E07070' if v < 0 else '#5DDAB4' for v in corr]
ax.barh(corr.index, corr.values, color=colors, edgecolor='none')
ax.axvline(0, color='white', lw=0.8)
ax.set_xlabel('Pearson r with logS')
ax.set_title('Descriptor correlations with solubility')
plt.tight_layout()
plt.show()

In [ ]:
ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  Ridge(alpha=1.0))
])
ridge.fit(X_desc_train, df_train['logS'])

y_pred_ridge = ridge.predict(X_desc_test)
rmse_ridge = mean_squared_error(df_test['logS'], y_pred_ridge, squared=False)
r2_ridge   = r2_score(df_test['logS'], y_pred_ridge)

print(f"Ridge regression (descriptors) — Test RMSE: {rmse_ridge:.3f}  R²: {r2_ridge:.3f}")

---
## 3. Representation 2 — Morgan fingerprints (ECFP4)

Morgan fingerprints encode the **local chemical environment** of each atom out to a given radius. With radius=2 and 2048 bits they are known as **ECFP4**.

```
Molecule → iterate atom environments (radius 0,1,2) → hash → fold to 2048-bit vector
```

Key properties:
- **Fixed-length** bit vector (or count vector)
- **No explicit bond-order/chirality** by default (configurable)
- **Sparse** — typically <5% of bits set
- Highly effective for similarity search and ML baselines

In [ ]:
def morgan_matrix(smiles_list, radius=2, n_bits=2048):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
            arr = np.zeros(n_bits, dtype=np.uint8)
            DataStructs.ConvertToNumpyArray(fp, arr)
            fps.append(arr)
        else:
            fps.append(np.zeros(n_bits, dtype=np.uint8))
    return np.array(fps)

X_fp_train = morgan_matrix(df_train['smiles'])
X_fp_val   = morgan_matrix(df_val['smiles'])
X_fp_test  = morgan_matrix(df_test['smiles'])

print(f"Fingerprint matrix shape: {X_fp_train.shape}")
print(f"Average bit density: {X_fp_train.mean()*100:.1f}%")

In [ ]:
# Visualise the fingerprint sparsity for the first 20 molecules
fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(X_fp_train[:20], aspect='auto', cmap='Greens', interpolation='nearest')
ax.set_xlabel('Bit index (2048)')
ax.set_ylabel('Molecule')
ax.set_title('ECFP4 fingerprints — first 20 molecules (green = bit set)')
plt.tight_layout()
plt.show()

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_features=0.3, random_state=42, n_jobs=-1)
rf.fit(X_fp_train, df_train['logS'])

y_pred_rf = rf.predict(X_fp_test)
rmse_rf = mean_squared_error(df_test['logS'], y_pred_rf, squared=False)
r2_rf   = r2_score(df_test['logS'], y_pred_rf)

print(f"Random Forest (ECFP4) — Test RMSE: {rmse_rf:.3f}  R²: {r2_rf:.3f}")

---
## 4. Representation 2b — Morgan fingerprints + neural network

Same input, different model. A fully-connected network can learn **non-linear feature interactions** that the random forest approximates through splits.

> Can a deeper model extract more information from the same fingerprint?

In [ ]:
class FingerprintNet(nn.Module):
    def __init__(self, in_dim=2048, hidden=256, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def make_tensor_loader(X, y, batch_size=64, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)

train_loader = make_tensor_loader(X_fp_train, df_train['logS'].values)
val_loader   = make_tensor_loader(X_fp_val,   df_val['logS'].values,   shuffle=False)
test_loader  = make_tensor_loader(X_fp_test,  df_test['logS'].values,  shuffle=False)

model_fp = FingerprintNet()
optimizer = torch.optim.Adam(model_fp.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

train_losses, val_losses = [], []

for epoch in range(80):
    model_fp.train()
    epoch_loss = 0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model_fp(Xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(yb)
    train_losses.append((epoch_loss / len(df_train)) ** 0.5)

    model_fp.eval()
    with torch.no_grad():
        val_preds = torch.cat([model_fp(Xb) for Xb, _ in val_loader])
        val_true  = torch.tensor(df_val['logS'].values, dtype=torch.float32)
        val_losses.append(criterion(val_preds, val_true).item() ** 0.5)

print(f"Final val RMSE: {val_losses[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label='Train RMSE')
ax.plot(val_losses,   label='Val RMSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE')
ax.set_title('Training curve — NN on ECFP4')
ax.legend()
plt.tight_layout()
plt.show()

model_fp.eval()
with torch.no_grad():
    y_pred_nn_fp = model_fp(torch.tensor(X_fp_test, dtype=torch.float32)).numpy()
rmse_nn_fp = mean_squared_error(df_test['logS'], y_pred_nn_fp, squared=False)
r2_nn_fp   = r2_score(df_test['logS'], y_pred_nn_fp)
print(f"NN (ECFP4) — Test RMSE: {rmse_nn_fp:.3f}  R²: {r2_nn_fp:.3f}")

---
## 5. Representation 3 — Molecular graph

Instead of hashing atom environments into a fixed vector, we represent the molecule **as-is**: atoms are nodes, bonds are edges.

```
Molecule → atoms (node features) + bonds (edge indices) → Graph → GNN → property
```

### Node features (per atom)
| Feature | Values |
|---------|--------|
| Atomic number | one-hot (C, N, O, S, F, Cl, Br, other) |
| Degree | 0–5 |
| Formal charge | int |
| Hybridization | SP, SP2, SP3 |
| Aromaticity | 0/1 |
| Num H | 0–4 |

The GNN uses **graph convolutions** to aggregate neighborhood information, then a global pooling to get a fixed-size molecule embedding.

> **Key difference from fingerprints:** the GNN *learns* which structural patterns matter for the property, rather than using predetermined hash buckets.

In [ ]:
ATOM_TYPES = ['C', 'N', 'O', 'S', 'F', 'Cl', 'Br', 'I', 'P', 'other']

def atom_features(atom):
    """Return a 1D feature vector for a single atom."""
    # Atom type one-hot
    sym = atom.GetSymbol()
    atom_oh = [int(sym == t) for t in ATOM_TYPES[:-1]] + [int(sym not in ATOM_TYPES[:-1])]
    # Degree (capped at 5)
    degree_oh = [int(atom.GetDegree() == d) for d in range(6)]
    # Hybridization
    hyb = atom.GetHybridization()
    hyb_oh = [
        int(hyb == Chem.rdchem.HybridizationType.SP),
        int(hyb == Chem.rdchem.HybridizationType.SP2),
        int(hyb == Chem.rdchem.HybridizationType.SP3),
    ]
    # Scalar features
    scalars = [
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        atom.GetTotalNumHs(),
    ]
    return atom_oh + degree_oh + hyb_oh + scalars

NODE_DIM = len(atom_features(Chem.MolFromSmiles('C').GetAtomWithIdx(0)))
print(f"Node feature dimension: {NODE_DIM}")

def smiles_to_graph(smi, y_val):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    # Node features
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    # Edge indices (undirected → add both directions)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    if edges:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    y = torch.tensor([y_val], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, y=y)

# Build graph datasets
graph_train = [g for g in (smiles_to_graph(s, y) for s, y in zip(df_train['smiles'], df_train['logS'])) if g]
graph_val   = [g for g in (smiles_to_graph(s, y) for s, y in zip(df_val['smiles'],   df_val['logS']))   if g]
graph_test  = [g for g in (smiles_to_graph(s, y) for s, y in zip(df_test['smiles'],  df_test['logS']))  if g]

print(f"Graph train: {len(graph_train)}  val: {len(graph_val)}  test: {len(graph_test)}")
print(f"Example graph: {graph_train[0]}")

In [ ]:
class GCN(nn.Module):
    def __init__(self, node_dim=NODE_DIM, hidden=64):
        super().__init__()
        self.conv1 = GCNConv(node_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.conv3 = GCNConv(hidden, hidden)
        self.head  = nn.Sequential(
            nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1)
        )

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        x = global_mean_pool(x, batch)   # aggregate atoms → molecule
        return self.head(x).squeeze(-1)

gnn_train_loader = GeoLoader(graph_train, batch_size=32, shuffle=True)
gnn_val_loader   = GeoLoader(graph_val,   batch_size=64)
gnn_test_loader  = GeoLoader(graph_test,  batch_size=64)

gnn = GCN()
gnn_opt = torch.optim.Adam(gnn.parameters(), lr=1e-3, weight_decay=1e-4)

gnn_train_losses, gnn_val_losses = [], []

for epoch in range(100):
    gnn.train()
    epoch_loss = 0
    for batch in gnn_train_loader:
        gnn_opt.zero_grad()
        pred = gnn(batch)
        loss = F.mse_loss(pred, batch.y)
        loss.backward()
        gnn_opt.step()
        epoch_loss += loss.item() * batch.num_graphs
    gnn_train_losses.append((epoch_loss / len(graph_train)) ** 0.5)

    gnn.eval()
    with torch.no_grad():
        val_pred = torch.cat([gnn(b) for b in gnn_val_loader])
        val_true = torch.cat([b.y for b in gnn_val_loader])
        gnn_val_losses.append(F.mse_loss(val_pred, val_true).item() ** 0.5)

print(f"GNN — Final val RMSE: {gnn_val_losses[-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gnn_train_losses, label='Train RMSE')
ax.plot(gnn_val_losses,   label='Val RMSE')
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE')
ax.set_title('Training curve — GCN on molecular graphs')
ax.legend()
plt.tight_layout()
plt.show()

gnn.eval()
with torch.no_grad():
    y_pred_gnn = torch.cat([gnn(b) for b in gnn_test_loader]).numpy()
y_true_test = df_test['logS'].values[:len(y_pred_gnn)]
rmse_gnn = mean_squared_error(y_true_test, y_pred_gnn, squared=False)
r2_gnn   = r2_score(y_true_test, y_pred_gnn)
print(f"GCN — Test RMSE: {rmse_gnn:.3f}  R²: {r2_gnn:.3f}")

---
## 6. Comparison & Discussion

In [ ]:
results = pd.DataFrame([
    {'Model': 'Ridge (descriptors)',  'Representation': 'Physicochemical', 'RMSE': rmse_ridge, 'R²': r2_ridge},
    {'Model': 'Random Forest (ECFP4)', 'Representation': 'Morgan fingerprint', 'RMSE': rmse_rf,    'R²': r2_rf},
    {'Model': 'NN (ECFP4)',            'Representation': 'Morgan fingerprint', 'RMSE': rmse_nn_fp, 'R²': r2_nn_fp},
    {'Model': 'GCN (graph)',           'Representation': 'Molecular graph',   'RMSE': rmse_gnn,   'R²': r2_gnn},
])

print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colors = ['#A8E8D4', '#5DDAB4', '#2DB890', '#0F6E56']

axes[0].bar(results['Model'], results['RMSE'], color=colors, edgecolor='none')
axes[0].set_ylabel('Test RMSE (lower is better)')
axes[0].set_title('RMSE by model')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(results['Model'], results['R²'], color=colors, edgecolor='none')
axes[1].set_ylabel('Test R² (higher is better)')
axes[1].set_title('R² by model')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Parity plots for all four models
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
y_true = df_test['logS'].values

pairs = [
    ('Ridge\n(descriptors)',  y_pred_ridge),
    ('RF\n(ECFP4)',           y_pred_rf),
    ('NN\n(ECFP4)',           y_pred_nn_fp),
    ('GCN\n(graph)',          y_pred_gnn),
]

for ax, (title, y_pred) in zip(axes, pairs):
    n = min(len(y_true), len(y_pred))
    ax.scatter(y_true[:n], y_pred[:n], alpha=0.5, s=18, c='#5DDAB4')
    lim = [min(y_true[:n].min(), y_pred[:n].min()) - 0.3,
           max(y_true[:n].max(), y_pred[:n].max()) + 0.3]
    ax.plot(lim, lim, 'w--', lw=1)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_xlabel('Measured logS')
    ax.set_title(title)

axes[0].set_ylabel('Predicted logS')
plt.suptitle('Parity plots — test set', y=1.02)
plt.tight_layout()
plt.show()

### Discussion

| Representation | Strengths | Limitations |
|---------------|-----------|-------------|
| Physicochemical descriptors | Interpretable, fast, domain-informed | Fixed feature set; misses structural patterns not encoded |
| Morgan fingerprints | Captures local environments, widely validated | Bit collisions; no spatial awareness; fixed radius |
| Graph (GCN) | Flexible, learns task-relevant patterns, no information loss | Needs more data; harder to interpret; slower to train |

**Scaffold split note:** we used a scaffold split (Bemis-Murcko), which is harder than random split — models must generalise to new chemical scaffolds. RMSE values are therefore higher than often reported in the literature.

---
## 7. Exercises

### Easy
1. Change the Morgan fingerprint radius from 2 to 3 (ECFP6). Does the Random Forest improve?
2. Add `NumAromaticRings` and `NumAliphaticRings` to the descriptor set. How does Ridge regression change?

### Medium
3. The GCN uses no edge features — bonds are just adjacency. Add bond-type features (single/double/aromatic) and check whether RMSE improves. *(Hint: use `GCNConv` → `NNConv` or add bond features to `smiles_to_graph`.)*
4. Replace `global_mean_pool` with `global_max_pool` or `global_add_pool`. Which performs best and why?

### Challenge
5. Compute a **Tanimoto similarity matrix** between the 30 least-soluble molecules and the 30 most-soluble ones using ECFP4. Visualise as a heatmap. What does it tell you about the chemical space of solubility?
6. Retrain all four models on a **random** train/test split instead of a scaffold split. How much does the RMSE change? What does this tell you about evaluating ML models in chemistry?

In [ ]:
# Your code here


---
## Further Reading

- **Rogers & Hahn (2010)** — *Extended-Connectivity Fingerprints* — J. Chem. Inf. Model. 50, 742 — the ECFP reference
- **Duvenaud et al. (2015)** — *Convolutional Networks on Graphs for Learning Molecular Fingerprints* — NeurIPS
- **Yang et al. (2019)** — *Analyzing Learned Molecular Representations for Property Prediction* — J. Chem. Inf. Model. 59, 3370 (Chemprop paper — benchmarks fingerprints vs graphs on MoleculeNet)
- **David et al. (2020)** — *Molecular Representations in AI-driven Drug Discovery* — J. Cheminform. 12, 56 — accessible review
- **Delaney (2004)** — *ESOL: Estimating Aqueous Solubility Directly from Molecular Structure* — J. Chem. Inf. Comput. Sci. 44, 1000 — original dataset paper